# Video Saturation Sampling

Sample up to 30 frames per movie at 5-second intervals, measure HSV saturation, and save movie-level plus frame-level CSV files. Progress is saved at movie granularity so the notebook can be stopped and resumed safely.

In [1]:
# Dependency check. Set AUTO_INSTALL_DEPENDENCIES=False if you prefer to install manually.
AUTO_INSTALL_DEPENDENCIES = True

import importlib
import subprocess
import sys

required_modules = {
    "cv2": "opencv-python-headless",
    "numpy": "numpy",
    "pandas": "pandas",
    "tqdm": "tqdm",
}

missing_packages = []
for module_name, package_name in required_modules.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_name)

if missing_packages and AUTO_INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *missing_packages])
    missing_packages = []
    for module_name, package_name in required_modules.items():
        try:
            importlib.import_module(module_name)
        except ModuleNotFoundError:
            missing_packages.append(package_name)

if missing_packages:
    packages = " ".join(missing_packages)
    raise ModuleNotFoundError(
        "Missing package(s): " + packages + "\n\n"
        "This Jupyter kernel is running:\n"
        f"  {sys.executable}\n\n"
        "Install into this exact kernel with a notebook cell like:\n"
        f"  import sys\n  !{{sys.executable}} -m pip install -U {packages}\n"
    )

print("Dependencies look ready.")


Dependencies look ready.


In [2]:
from __future__ import annotations

import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

VIDEO_DIR = Path("/Volumes/Extreme SSD/Video/Chinese1949-66")
OUTPUT_DIR = Path("./saturation_sample_results")

VIDEO_EXTENSIONS = {".mp4", ".mov", ".m4v", ".avi", ".mkv"}
START_TIME_SEC = 0.0
SAMPLE_INTERVAL_SEC = 5.0
MAX_SAMPLED_FRAMES = 30
RESIZE_WIDTH = 320
ROUND_DIGITS = 4
FRAME_READER_BACKEND = "ffmpeg_pipe"  # "ffmpeg_pipe", "opencv_seek", or "auto". ffmpeg_pipe avoids 30 separate seeks.
SATURATION_BACKEND = "auto"  # "auto", "cpu", or "mps". MPS needs an existing PyTorch install.
FFMPEG_BINARY = "ffmpeg"

# Skip movies already marked complete when file size and mtime still match.
FORCE_RECOMPUTE = False

SUMMARY_CSV = OUTPUT_DIR / "movie_saturation_summary.csv"
FRAMES_CSV = OUTPUT_DIR / "movie_saturation_frames.csv"
PROGRESS_CSV = OUTPUT_DIR / "movie_saturation_progress.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Video dir: {VIDEO_DIR.resolve()}")
print(f"Output dir: {OUTPUT_DIR.resolve()}")


Video dir: /Volumes/Extreme SSD/Video/Chinese1949-66
Output dir: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/saturation_sample_results


/Users/caoshuangyuan/anaconda3/envs/coreml4/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SUMMARY_COLUMNS = [
    "film",
    "video_path",
    "file_size",
    "mtime_ns",
    "sampled_frame_count",
    "requested_frame_count",
    "start_time_sec",
    "sample_interval_sec",
    "max_sampled_frames",
    "video_fps",
    "video_frame_count",
    "video_duration_sec",
    "max_saturation_any_pixel",
    "max_frame_mean_saturation",
    "mean_saturation_30_frames",
    "median_saturation_30_frames",
    "mean_frame_median_saturation",
    "median_frame_median_saturation",
    "created_at",
]

FRAME_COLUMNS = [
    "film",
    "video_path",
    "frame_sample_index",
    "timestamp_sec",
    "read_ok",
    "saturation_mean",
    "saturation_median",
    "saturation_max",
    "frame_width",
    "frame_height",
]

PROGRESS_COLUMNS = [
    "film",
    "video_path",
    "file_size",
    "mtime_ns",
    "status",
    "sampled_frame_count",
    "error",
    "started_at",
    "finished_at",
]


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def load_csv(path: Path, columns: list[str]) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=columns)
    return pd.read_csv(path)


def atomic_write_csv(df: pd.DataFrame, path: Path) -> None:
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, path)


def video_file_info(path: Path) -> dict:
    stat = path.stat()
    return {
        "film": path.stem,
        "video_path": str(path),
        "file_size": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
    }


def discover_videos(video_dir: Path) -> list[Path]:
    if not video_dir.exists():
        raise FileNotFoundError(f"Video directory does not exist: {video_dir}")
    videos = []
    for path in video_dir.rglob("*"):
        if not path.is_file():
            continue
        if path.name.startswith("._"):
            continue
        if path.suffix.lower() in VIDEO_EXTENSIONS:
            videos.append(path)
    return sorted(videos, key=lambda p: p.name)


def is_complete_and_current(progress_df: pd.DataFrame, path: Path) -> bool:
    if FORCE_RECOMPUTE or progress_df.empty:
        return False
    info = video_file_info(path)
    rows = progress_df[progress_df["video_path"] == info["video_path"]]
    if rows.empty:
        return False
    row = rows.iloc[-1]
    return (
        row.get("status") == "complete"
        and int(row.get("file_size", -1)) == info["file_size"]
        and int(row.get("mtime_ns", -1)) == info["mtime_ns"]
    )


def replace_rows_by_video_path(df: pd.DataFrame, video_path: str, new_rows: list[dict], columns: list[str]) -> pd.DataFrame:
    if df.empty:
        base = pd.DataFrame(columns=columns)
    else:
        base = df[df["video_path"] != video_path].copy()
    if new_rows:
        base = pd.concat([base, pd.DataFrame(new_rows)], ignore_index=True)
    return base.reindex(columns=columns)


def upsert_progress(progress_df: pd.DataFrame, row: dict) -> pd.DataFrame:
    return replace_rows_by_video_path(progress_df, row["video_path"], [row], PROGRESS_COLUMNS)


In [4]:
def resize_for_stats(frame: np.ndarray, resize_width: Optional[int]) -> np.ndarray:
    if not resize_width or resize_width <= 0:
        return frame
    height, width = frame.shape[:2]
    if width <= resize_width:
        return frame
    scale = resize_width / float(width)
    new_size = (resize_width, max(1, int(round(height * scale))))
    return cv2.resize(frame, new_size, interpolation=cv2.INTER_AREA)


def read_frame_at_time(cap: cv2.VideoCapture, timestamp_sec: float, fps: Optional[float]) -> tuple[bool, Optional[np.ndarray]]:
    cap.set(cv2.CAP_PROP_POS_MSEC, float(timestamp_sec) * 1000.0)
    ok, frame = cap.read()
    if ok:
        return True, frame
    if fps and fps > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(round(timestamp_sec * fps)))
        ok, frame = cap.read()
        if ok:
            return True, frame
    return False, None


def resolve_frame_reader_backend() -> str:
    requested = str(FRAME_READER_BACKEND).lower().strip()
    if requested not in {"auto", "ffmpeg_pipe", "opencv_seek"}:
        raise ValueError('FRAME_READER_BACKEND must be "auto", "ffmpeg_pipe", or "opencv_seek"')
    ffmpeg_exists = shutil.which(FFMPEG_BINARY) is not None
    if requested == "ffmpeg_pipe" and not ffmpeg_exists:
        raise RuntimeError(f"FFmpeg binary not found: {FFMPEG_BINARY}")
    if requested == "auto":
        return "ffmpeg_pipe" if ffmpeg_exists else "opencv_seek"
    return requested


def resolve_saturation_backend() -> str:
    requested = str(SATURATION_BACKEND).lower().strip()
    if requested not in {"auto", "cpu", "mps"}:
        raise ValueError('SATURATION_BACKEND must be "auto", "cpu", or "mps"')
    if requested == "cpu":
        return "cpu"
    try:
        import torch

        mps_available = bool(torch.backends.mps.is_available())
    except Exception:
        if requested == "mps":
            raise RuntimeError("SATURATION_BACKEND='mps' requires PyTorch with MPS support in this kernel.")
        return "cpu"
    if requested == "mps" and not mps_available:
        raise RuntimeError("PyTorch is installed, but torch.backends.mps.is_available() is False.")
    return "mps" if mps_available else "cpu"


def saturation_from_rgb_array_cpu(frames_rgb: np.ndarray) -> np.ndarray:
    pixels = frames_rgb.astype(np.float32)
    channel_max = pixels.max(axis=-1)
    channel_min = pixels.min(axis=-1)
    saturation = np.zeros_like(channel_max, dtype=np.float32)
    np.divide(channel_max - channel_min, channel_max, out=saturation, where=channel_max > 0)
    return saturation * 255.0


def saturation_stats_batch_cpu(frames_rgb: list[np.ndarray]) -> list[dict]:
    resized_frames = [resize_for_stats(frame, RESIZE_WIDTH) for frame in frames_rgb]
    frames = np.stack(resized_frames, axis=0)
    saturation = saturation_from_rgb_array_cpu(frames)
    flat = saturation.reshape((saturation.shape[0], -1))
    rows = []
    for idx, frame in enumerate(resized_frames):
        height, width = frame.shape[:2]
        rows.append(
            {
                "saturation_mean": round(float(flat[idx].mean()), ROUND_DIGITS),
                "saturation_median": round(float(np.median(flat[idx])), ROUND_DIGITS),
                "saturation_max": round(float(flat[idx].max()), ROUND_DIGITS),
                "frame_width": int(width),
                "frame_height": int(height),
            }
        )
    return rows


def saturation_stats_batch_mps(frames_rgb: list[np.ndarray]) -> list[dict]:
    import torch

    resized_frames = [resize_for_stats(frame, RESIZE_WIDTH) for frame in frames_rgb]
    frames = np.stack(resized_frames, axis=0)
    tensor = torch.from_numpy(frames).to(device="mps", dtype=torch.float32)
    channel_max = tensor.amax(dim=-1)
    channel_min = tensor.amin(dim=-1)
    saturation = torch.where(channel_max > 0, (channel_max - channel_min) / channel_max * 255.0, torch.zeros_like(channel_max))
    flat = saturation.reshape((saturation.shape[0], -1))
    means = flat.mean(dim=1).detach().cpu().numpy()
    maxes = flat.amax(dim=1).detach().cpu().numpy()
    # Median on MPS can vary by PyTorch version; CPU median is cheap for only 30 resized frames.
    medians = np.median(flat.detach().cpu().numpy(), axis=1)

    rows = []
    for idx, frame in enumerate(resized_frames):
        height, width = frame.shape[:2]
        rows.append(
            {
                "saturation_mean": round(float(means[idx]), ROUND_DIGITS),
                "saturation_median": round(float(medians[idx]), ROUND_DIGITS),
                "saturation_max": round(float(maxes[idx]), ROUND_DIGITS),
                "frame_width": int(width),
                "frame_height": int(height),
            }
        )
    return rows


def saturation_stats_batch(frames_rgb: list[np.ndarray]) -> list[dict]:
    if not frames_rgb:
        return []
    backend = resolve_saturation_backend()
    if backend == "mps":
        try:
            return saturation_stats_batch_mps(frames_rgb)
        except Exception as exc:
            if str(SATURATION_BACKEND).lower().strip() == "mps":
                raise
            print(f"MPS saturation failed; falling back to CPU for this batch: {exc}")
    return saturation_stats_batch_cpu(frames_rgb)


def read_ppm_token(stream) -> bytes:
    token = bytearray()
    while True:
        char = stream.read(1)
        if not char:
            return bytes(token)
        if char == b"#":
            stream.readline()
            continue
        if char.isspace():
            if token:
                return bytes(token)
            continue
        token.extend(char)


def iter_ppm_frames(stream):
    while True:
        magic = read_ppm_token(stream)
        if not magic:
            return
        if magic != b"P6":
            raise RuntimeError(f"Unexpected PPM magic from ffmpeg: {magic!r}")
        width = int(read_ppm_token(stream))
        height = int(read_ppm_token(stream))
        maxval = int(read_ppm_token(stream))
        if maxval != 255:
            raise RuntimeError(f"Unexpected PPM max value from ffmpeg: {maxval}")
        byte_count = width * height * 3
        payload = stream.read(byte_count)
        if len(payload) != byte_count:
            raise RuntimeError("Unexpected end of ffmpeg PPM frame payload")
        frame = np.frombuffer(payload, dtype=np.uint8).reshape((height, width, 3)).copy()
        yield frame


def sample_frames_ffmpeg_pipe(path: Path) -> tuple[list[np.ndarray], float, int, float]:
    sample_duration_sec = START_TIME_SEC + SAMPLE_INTERVAL_SEC * max(MAX_SAMPLED_FRAMES, 1)
    select_filter = f"select='isnan(prev_selected_t)+gte(t-prev_selected_t,{SAMPLE_INTERVAL_SEC})'"
    vf_filter = f"{select_filter},scale={RESIZE_WIDTH}:-2"
    command = [
        FFMPEG_BINARY,
        "-hide_banner",
        "-loglevel",
        "error",
        "-nostdin",
    ]
    if START_TIME_SEC > 0:
        command += ["-ss", str(float(START_TIME_SEC))]
    command += [
        "-t",
        str(float(sample_duration_sec)),
        "-i",
        str(path),
        "-vf",
        vf_filter,
        "-vsync",
        "0",
        "-frames:v",
        str(int(MAX_SAMPLED_FRAMES)),
        "-an",
        "-sn",
        "-f",
        "image2pipe",
        "-vcodec",
        "ppm",
        "pipe:1",
    ]
    proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    assert proc.stdout is not None
    frames = list(iter_ppm_frames(proc.stdout))
    stderr = proc.stderr.read().decode("utf-8", errors="replace") if proc.stderr else ""
    return_code = proc.wait()
    if return_code != 0:
        raise RuntimeError(f"ffmpeg failed for {path.name}: {stderr.strip()}")
    if not frames:
        raise RuntimeError(f"ffmpeg returned no sampled frames for {path}")
    # We keep these metadata fields via OpenCV because they are convenient and not performance-critical.
    cap = cv2.VideoCapture(str(path))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0) if cap.isOpened() else 0.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0) if cap.isOpened() else 0
    cap.release()
    duration_sec = round(frame_count / fps, ROUND_DIGITS) if fps > 0 and frame_count > 0 else np.nan
    return frames, fps, frame_count, duration_sec


def sample_frames_opencv_seek(path: Path) -> tuple[list[np.ndarray], list[bool], float, int, float]:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {path}")
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration_sec = round(frame_count / fps, ROUND_DIGITS) if fps > 0 and frame_count > 0 else np.nan
    frames = []
    read_flags = []
    try:
        for timestamp_sec in [START_TIME_SEC + i * SAMPLE_INTERVAL_SEC for i in range(MAX_SAMPLED_FRAMES)]:
            ok, frame_bgr = read_frame_at_time(cap, timestamp_sec, fps)
            read_flags.append(bool(ok))
            if ok and frame_bgr is not None:
                frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    finally:
        cap.release()
    return frames, read_flags, fps, frame_count, duration_sec


def sample_video_saturation(path: Path) -> tuple[list[dict], dict]:
    info = video_file_info(path)
    timestamps = [START_TIME_SEC + i * SAMPLE_INTERVAL_SEC for i in range(MAX_SAMPLED_FRAMES)]
    frame_rows = [
        {
            "film": info["film"],
            "video_path": info["video_path"],
            "frame_sample_index": sample_index,
            "timestamp_sec": round(float(timestamp_sec), ROUND_DIGITS),
            "read_ok": False,
            "saturation_mean": np.nan,
            "saturation_median": np.nan,
            "saturation_max": np.nan,
            "frame_width": np.nan,
            "frame_height": np.nan,
        }
        for sample_index, timestamp_sec in enumerate(timestamps, start=1)
    ]

    reader_backend = resolve_frame_reader_backend()
    if reader_backend == "ffmpeg_pipe":
        frames, fps, frame_count, duration_sec = sample_frames_ffmpeg_pipe(path)
        read_flags = [True] * len(frames) + [False] * max(0, MAX_SAMPLED_FRAMES - len(frames))
    else:
        frames, read_flags, fps, frame_count, duration_sec = sample_frames_opencv_seek(path)

    if not frames:
        raise RuntimeError(f"No sampled frames could be read from video: {path}")

    stats_rows = saturation_stats_batch(frames)
    stats_index = 0
    for row, read_ok in zip(frame_rows, read_flags):
        row["read_ok"] = bool(read_ok)
        if read_ok and stats_index < len(stats_rows):
            row.update(stats_rows[stats_index])
            stats_index += 1

    valid = pd.DataFrame(frame_rows)
    valid = valid[valid["read_ok"] == True].copy()
    if valid.empty:
        raise RuntimeError(f"No sampled frames could be read from video: {path}")

    summary = {
        **info,
        "sampled_frame_count": int(len(valid)),
        "requested_frame_count": int(MAX_SAMPLED_FRAMES),
        "start_time_sec": float(START_TIME_SEC),
        "sample_interval_sec": float(SAMPLE_INTERVAL_SEC),
        "max_sampled_frames": int(MAX_SAMPLED_FRAMES),
        "video_fps": round(fps, ROUND_DIGITS) if fps else np.nan,
        "video_frame_count": int(frame_count),
        "video_duration_sec": duration_sec,
        "max_saturation_any_pixel": round(float(valid["saturation_max"].max()), ROUND_DIGITS),
        "max_frame_mean_saturation": round(float(valid["saturation_mean"].max()), ROUND_DIGITS),
        "mean_saturation_30_frames": round(float(valid["saturation_mean"].mean()), ROUND_DIGITS),
        "median_saturation_30_frames": round(float(valid["saturation_mean"].median()), ROUND_DIGITS),
        "mean_frame_median_saturation": round(float(valid["saturation_median"].mean()), ROUND_DIGITS),
        "median_frame_median_saturation": round(float(valid["saturation_median"].median()), ROUND_DIGITS),
        "created_at": now_iso(),
    }
    return frame_rows, summary


print(
    f"Frame reader backend: {resolve_frame_reader_backend()} | "
    f"Saturation stats backend: {resolve_saturation_backend()} "
    "(FFmpeg avoids repeated OpenCV seeks; decode still runs outside Metal)"
)


In [5]:
videos = discover_videos(VIDEO_DIR)
progress_df = load_csv(PROGRESS_CSV, PROGRESS_COLUMNS)
summary_df = load_csv(SUMMARY_CSV, SUMMARY_COLUMNS)
frames_df = load_csv(FRAMES_CSV, FRAME_COLUMNS)

pending_videos = [path for path in videos if not is_complete_and_current(progress_df, path)]

print(f"Discovered videos: {len(videos)}")
print(f"Already complete/current: {len(videos) - len(pending_videos)}")
print(f"Pending this run: {len(pending_videos)}")
print("First pending videos:")
for path in pending_videos[:10]:
    print("-", path.name)


FileNotFoundError: Video directory does not exist: /Volumes/Extreme SSD/Video/Chinese1949-66

In [ ]:
for video_path in tqdm(pending_videos, desc="Sampling saturation"):
    info = video_file_info(video_path)
    started_at = now_iso()
    progress_df = upsert_progress(
        progress_df,
        {
            **info,
            "status": "running",
            "sampled_frame_count": 0,
            "error": "",
            "started_at": started_at,
            "finished_at": "",
        },
    )
    atomic_write_csv(progress_df, PROGRESS_CSV)

    try:
        frame_rows, summary_row = sample_video_saturation(video_path)
        summary_df = replace_rows_by_video_path(summary_df, info["video_path"], [summary_row], SUMMARY_COLUMNS)
        frames_df = replace_rows_by_video_path(frames_df, info["video_path"], frame_rows, FRAME_COLUMNS)

        atomic_write_csv(summary_df.sort_values("film"), SUMMARY_CSV)
        atomic_write_csv(frames_df.sort_values(["film", "frame_sample_index"]), FRAMES_CSV)

        progress_df = upsert_progress(
            progress_df,
            {
                **info,
                "status": "complete",
                "sampled_frame_count": int(summary_row["sampled_frame_count"]),
                "error": "",
                "started_at": started_at,
                "finished_at": now_iso(),
            },
        )
        atomic_write_csv(progress_df.sort_values("film"), PROGRESS_CSV)
    except Exception as exc:
        progress_df = upsert_progress(
            progress_df,
            {
                **info,
                "status": "failed",
                "sampled_frame_count": 0,
                "error": str(exc),
                "started_at": started_at,
                "finished_at": now_iso(),
            },
        )
        atomic_write_csv(progress_df.sort_values("film"), PROGRESS_CSV)
        print(f"FAILED: {video_path.name}: {exc}")

print("Done.")


In [ ]:
summary_df = load_csv(SUMMARY_CSV, SUMMARY_COLUMNS)
progress_df = load_csv(PROGRESS_CSV, PROGRESS_COLUMNS)

completed = int((progress_df["status"] == "complete").sum()) if not progress_df.empty else 0
failed = int((progress_df["status"] == "failed").sum()) if not progress_df.empty else 0
running = int((progress_df["status"] == "running").sum()) if not progress_df.empty else 0
print(f"Completed: {completed}")
print(f"Failed: {failed}")
print(f"Running/interrupted: {running}")
print(f"Summary rows: {len(summary_df)}")

if not summary_df.empty:
    display(
        summary_df.sort_values("mean_saturation_30_frames", ascending=False)[
            [
                "film",
                "sampled_frame_count",
                "max_saturation_any_pixel",
                "max_frame_mean_saturation",
                "mean_saturation_30_frames",
                "median_saturation_30_frames",
                "mean_frame_median_saturation",
            ]
        ].head(30)
    )
